## Adjust or remove next cell as needed. 
It is here to make sure you point to the correct JAVA_HOME 

In [1]:
import os
os.environ['JAVA_HOME']='/opt/homebrew/opt/openjdk@11'

## Make sure your knnAssignment file is in the same folder as this notebook.

In [2]:
from capymoa.base import Classifier
from capymoa.evaluation import prequential_evaluation
from capymoa.datasets import Electricity, RBFm_100k

## You can either create new class in the notebook

In [3]:
from knnAssignment import SearchNeighbours, newWindow

class KNNBase2(Classifier):
    def __init__(self, schema, k=3, window_size=1000, search_method="LinearNNSearch"):
        super().__init__(schema)
        self.k = k
        self.window_size = window_size
        self.search = SearchNeighbours(search_method)
        self.window = newWindow(schema)

    def __str__(self):
        return "Base2 kNN"

    def train(self, instance):
        if self.window.size() >= self.window_size:
            self.window.remove_instance()
        self.window.add_instance(instance)
    
    def predict(self, instance):
        """
        Returns the k-NN predictions for a given instance.
        Uses mean/median for regression or majority vote for classification.
        """
        try:
            if self.window and self.window.size() > 0:
                # Find k nearest neighbors
                neighbors = self.search.do_search(self.window, instance, self.k)
                # Vote for the most common class among the k closest neighbors
                votes = [0] * (self.schema.get_num_classes())
                for i in range(neighbors.size()):
                    class_idx = int(neighbors.get_instance(i).classValue())
                    votes[class_idx] += 1
                return votes.index(max(votes))

        except Exception as e:
            print("Error: kNN search failed.", e)
            return 0

    def predict_proba(self, instance):
        pass

## Or directly import classes from the knnAssignment file

In [4]:
from knnAssignment import KNNBase, KNNBase3

## Create the streams, first time you use this, it will download them.

In [5]:
elec_stream = Electricity()
rbf_stream = RBFm_100k()
list_datasets = [elec_stream, rbf_stream]

## loop through datasets, using the provided class and the two copies

Same algorithm, same data, if you do not find same accuracy, something is weird.

In [6]:
for ds in list_datasets:
	print(f"Dataset: {ds}")
	# Base kNN
	bknn = KNNBase(schema=ds.get_schema(), k=3, window_size=50)
	result = prequential_evaluation(stream=ds, learner=bknn, max_instances=50000)
	print(f"{bknn} accuracy: {result.cumulative.accuracy()}, wallclock: {result.wallclock()}\n")
	# Base2 kNN
	bknn2 = KNNBase2(schema=ds.get_schema(), k=3, window_size=50)
	result = prequential_evaluation(stream=ds, learner=bknn2, max_instances=50000)
	print(f"{bknn2} accuracy: {result.cumulative.accuracy()}, wallclock: {result.wallclock()}\n")
	# Base3 kNN
	bknn3 = KNNBase3(schema=ds.get_schema(), k=3, window_size=50)
	result = prequential_evaluation(stream=ds, learner=bknn3, max_instances=50000)
	print(f"{bknn3} accuracy: {result.cumulative.accuracy()}, wallclock: {result.wallclock()}\n")

Dataset: Electricity
Base kNN accuracy: 86.88206214689266, wallclock: 3.391686201095581

Base2 kNN accuracy: 86.88206214689266, wallclock: 3.215153932571411

Base3 kNN accuracy: 86.88206214689266, wallclock: 3.2827649116516113

Dataset: RBFm_100k
Base kNN accuracy: 50.194, wallclock: 3.9866552352905273

Base2 kNN accuracy: 50.194, wallclock: 4.009953737258911

Base3 kNN accuracy: 50.194, wallclock: 3.992506980895996

